In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import torch
import numpy as np
from transformers import GPT2Tokenizer, GPT2LMHeadModel

In [ ]:
# Directories containing the medical notes
gpt_note_dir = 
llama_note_dir = 

In [ ]:

# Load GPT-2 tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()  # Set model to evaluation mode

def chunk_text(text, chunk_size=1024, overlap=200):
    """Splits text into overlapping chunks to avoid truncation issues."""
    tokens = tokenizer.encode(text, add_special_tokens=True)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + chunk_size, len(tokens))
        chunks.append(tokens[start:end])
        start += chunk_size - overlap  # Overlap chunks to maintain context
    return chunks

def calculate_perplexity(text):
    """Calculates perplexity for long text by splitting into chunks."""
    if not text.strip():
        return float("inf"), 0  # Return infinity if text is empty
    
    chunks = chunk_text(text)
    total_loss = 0
    total_tokens = 0

    for chunk in chunks:
        encodings = torch.tensor(chunk).unsqueeze(0)  # Convert to tensor
        with torch.no_grad():
            outputs = model(input_ids=encodings, labels=encodings)
            loss = outputs.loss
            chunk_perplexity = torch.exp(loss).item()
        
        total_loss += loss.item() * len(chunk)  # Weight by token count
        total_tokens += len(chunk)
    
    avg_perplexity = torch.exp(torch.tensor(total_loss / total_tokens)).item() if total_tokens > 0 else float("inf")

    return avg_perplexity, total_tokens

def evaluate_perplexity(directory, source_name):
    """Calculates perplexity for each text file without truncation."""
    data = []
    for filename in os.listdir(directory):
        if filename.endswith(".txt"):
            file_path = os.path.join(directory, filename)
            with open(file_path, "r", encoding="utf-8",  errors="ignore") as file:
                text = file.read()
                perplexity, num_tokens = calculate_perplexity(text)
                perplexity_per_token = perplexity / num_tokens if num_tokens > 0 else float("inf")
                data.append({
                    "Filename": filename,
                    "Perplexity": perplexity,
                    "Tokens": num_tokens,
                    "Perplexity Per Token": perplexity_per_token,
                    "Source": source_name
                })
    
    return pd.DataFrame(data)

# Calculate perplexity without truncation
df_gpt = evaluate_perplexity(gpt_note_dir, "GPT")
df_llama = evaluate_perplexity(llama_note_dir, "LLaMA")

# Combine results
df_perplexity = pd.concat([df_gpt, df_llama], ignore_index=True)


In [ ]:
df_perplexity["Filename"] = df_perplexity["Filename"].str[:16]
df = df_perplexity.pivot(index="Filename", columns="Source", values="Perplexity").reset_index()
df.columns = ["Filename", "GPT Perplexity", "LLaMA Perplexity"]  # Rename columns

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['GPT Perplexity'], bins=10, alpha=0.5, label='GPT', color='blue', edgecolor='black')
plt.hist(df['LLaMA Perplexity'], bins=10, alpha=0.5, label='LLaMA', color='orange', edgecolor='black')
plt.title('Distribution of Perplexity')
plt.xlabel('Perplexity scores')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()

In [ ]:
# Define the save directory and file path
save_dir = 
save_path = os.path.join(save_dir, "Perplexity_Distribution.png")

# Create the histogram plot
plt.figure(figsize=(10, 6))
plt.hist(df['GPT Perplexity'], bins=10, alpha=0.5, label='GPT', color='blue', edgecolor='black')
plt.hist(df['LLaMA Perplexity'], bins=10, alpha=0.5, label='LLaMA', color='orange', edgecolor='black')

# Add labels and legend
plt.xlabel('Perplexity scores', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.legend(loc='upper right')

# Save the figure at 300 DPI
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"Figure saved successfully at: {save_path}")
plt.show()

In [ ]:
print(f"Perplexity for GPT is {df['GPT Perplexity'].mean()}")
print(f"Perplexity for LLaMA is {df['LLaMA Perplexity'].mean()}")


In [ ]:
import scipy.stats as stats

# Conduct paired t-test
t_stat, p_value = stats.ttest_rel(df['GPT Perplexity'], df['LLaMA Perplexity'])

# Print results
print(f"Paired t-test results: t-statistic = {t_stat:.4f}, p-value = {p_value:.4f}")


In [ ]:
df.head()

## Using Result from Rui

In [ ]:
file_path =
df = pd.read_csv(file_path)


In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['gpt_perplexity'], bins=10, alpha=0.5, label='GPT', color='blue', edgecolor='black')
plt.hist(df['llama_perplexity'], bins=10, alpha=0.5, label='LLaMA', color='orange', edgecolor='black')
plt.title('Distribution of Perplexity')
plt.xlabel('Perplexity Scores')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()

In [ ]:
df.head(10)